# Prompt Strategy Lab - Demo

Side-by-side comparison of 9 prompt-engineering strategies on the same user prompt, using a local TinyLLaMA via Ollama.

**Prerequisites:**
1. `ollama pull tinyllama`
2. `pip install -e .[dev]` from the repo root.
3. Make sure the Ollama server is running (`ollama serve` if not auto-started).

> A clean, library-using demo of the packaged code. For the original full walkthrough this project was extracted from - including all model outputs and the exploratory iteration - see [`walkthrough.ipynb`](walkthrough.ipynb).


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))

import pandas as pd
from prompt_strategy_lab import compare_strategies, OllamaRunner, STRATEGIES, to_records  # noqa: E402
from prompt_strategy_lab.compare import to_records

print('Available strategies:')
for name, s in STRATEGIES.items():
    print(f'  - {name}: {s.description}')

Available strategies:
  - chain_of_thought: Instructs the model to reason step-by-step before answering.
  - reflection: Generates an answer, then asks the model to critique its own output.
  - alternative_approaches: Forces the model to produce multiple framings from different perspectives.
  - template: Constrains output to a fixed structural template.
  - comparative: Generates multiple candidate solutions and recommends the best.
  - fact_check_list: Separates verifiable facts from assumptions for each output item.
  - few_shot: Conditions the model with examples of the desired output style.
  - cot_with_reflection: Two-pass: silent planning, then silent reflection, then final answer.
  - self_consistency: Internally generates several candidates and returns only the best-merged result.


## Step 1 - Define the user prompt and the strategies to compare

Pick any prompt and any subset of strategies. Add or remove strategies and re-run.

In [2]:
USER_PROMPT = (
    'Propose a backend code structure for an AI-powered exam studying app '
    'for SAT, GRE, and Medical Exams.'
)

STRATEGIES_TO_RUN = [
    'few_shot',
    'chain_of_thought',
    'cot_with_reflection',
    'self_consistency',
]

## Step 2 - Inspect the augmented prompts before sending them

This is what the model will actually see. Each strategy wraps the same user prompt in a different scaffold.

In [3]:
for name in STRATEGIES_TO_RUN:
    strategy = STRATEGIES[name]
    print('=' * 80)
    print(f'STRATEGY: {name}')
    print('=' * 80)
    print(strategy.augment(USER_PROMPT)[:800])
    print('...')
    print()

STRATEGY: few_shot
You are a Solution Design Bot.

Here are examples of the kind of output expected:

Example A:
project/
  app/
    __init__.py
    models.py
    routes.py
    services.py
  tests/
    test_app.py
  requirements.txt
  run.py

Example B:
backend/
  src/
    api/
      controllers.py
      serializers.py
    core/
      config.py
      database.py
    features/
      users.py
      ai_engine.py
  tests/
    unit/
    integration/
  pyproject.toml

Now, in a similar style, address: Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams.

Include:
- Folder layout
- Key files with one-line purpose each
- Short description of each component
...

STRATEGY: chain_of_thought
You are a senior product strategist.
Think step by step about the user's request before answering.

Task: Propose a backend code structure for an AI-powered exam studying app for SAT, GRE, and Medical Exams.

Process:
1. Analyze the user's request and the target

## Step 3 - Run the comparison

Each strategy is sent through the same `OllamaRunner` with identical generation parameters, so the only variable is the prompt scaffold itself.

In [4]:
runner = OllamaRunner(model='tinyllama')

results = compare_strategies(
    user_prompt=USER_PROMPT,
    strategy_names=STRATEGIES_TO_RUN,
    runner=runner,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    num_predict=600,
    context_window=2048,
)

## Step 4 - Compare numerically (latency / tokens)

In [5]:
df = pd.DataFrame(to_records(results))
df[['strategy', 'tokens_used', 'latency_ms']].sort_values('latency_ms')

,strategy,tokens_used,latency_ms
2,cot_with_reflection,249,3586.4422
1,chain_of_thought,597,8000.4548
3,self_consistency,600,8243.1340
0,few_shot,326,22435.6303


## Step 5 - Compare qualitatively (the actual responses)

In [6]:
for r in results:
    print('=' * 80)
    print(f'{r.strategy}  |  {r.latency_ms:.0f}ms  |  {r.tokens_used} tokens')
    print('=' * 80)
    print(r.response)
    print()

few_shot  |  22436ms  |  326 tokens
Example A:
Project/
   App/
      __init__.py
      models.py
      routes.py
      services.py
   Tests/
     test_app.py
   Requirements.txt
   Run.py

Example B:
Backend/
   src/
      api/
         controller.py (for REST API)
         serializers.py (for JSON-API)
       features/
          users.py (for User registration and login)
          ai_engine.py (for AI engine module)
       tests/
         unit/
           test_controller.py (for unit testing controller)
         integration/
           test_serializer.py (for unit testing serializer)
   pyproject.toml (for dependency management and tooling)

In this example, we have divided the code into separate folders for better organization and modularity. The project directory structure includes an "App" folder with its own "src/" folder and each sub-folder containing the controllers, serializers, and features/modules as discussed above. Each module is self-contained with its own dedicated file 

## Step 6 - Save results for the README

In [7]:
from pathlib import Path
import datetime as dt

results_dir = Path('../results')
results_dir.mkdir(exist_ok=True)
stamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = results_dir / f'comparison_{stamp}.csv'
df.to_csv(out_path, index=False)
print(f'Saved {out_path}')

Saved ..\results\comparison_20260425_164951.csv
